In [0]:
# Configuration


CATALOG = "worldbank_ai"
BRONZE_SCHEMA = "bronze"

WORLD_BANK_API_BASE_URL = "https://api.worldbank.org/v2"

INDICATOR_METADATA_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.indicator_metadata_raw"
)

INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "GDP growth (annual %)",
    "NY.GDP.PCAP.KD.ZG": "GDP per capita growth (annual %)",
    "NY.GDP.MKTP.CD": "GDP (current US$)",
    "NY.GDP.PCAP.CD": "GDP per capita (current US$)",
    "FP.CPI.TOTL.ZG": "Inflation, consumer prices (annual %)",
    "NE.TRD.GNFS.ZS": "Trade (% of GDP)",
    "NE.EXP.GNFS.ZS": "Exports of goods and services (% of GDP)",
    "NE.IMP.GNFS.ZS": "Imports of goods and services (% of GDP)",
    "NE.GDI.TOTL.ZS": "Gross capital formation (% of GDP)",
    "GC.XPN.TOTL.GD.ZS": "Expense (% of GDP)",
    "GC.REV.XGRT.GD.ZS": "Revenue, excluding grants (% of GDP)",
    "GC.DOD.TOTL.GD.ZS": "Central government debt, total (% of GDP)",
    "NY.GDS.TOTL.ZS": "Gross domestic savings (% of GDP)",
    "BX.KLT.DINV.WD.GD.ZS":
        "Foreign direct investment, net inflows (% of GDP)",
    "BN.CAB.XOKA.GD.ZS": "Current account balance (% of GDP)"
}

print(f"Indicators configured: {len(INDICATORS)}")
print(f"Target table: {INDICATOR_METADATA_TABLE}")

Indicators configured: 15
Target table: worldbank_ai.bronze.indicator_metadata_raw


In [0]:
# Imports


import requests
import time
import json

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Imports loaded.")

Imports loaded.


In [0]:
# Reusable request function


def get_json_with_retry(
    url,
    params=None,
    max_retries=4,
    timeout=30
):
    retryable_status_codes = {
        429, 500, 502, 503, 504
    }

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=timeout
            )

            if response.status_code == 200:
                return response.json()

            if response.status_code in retryable_status_codes:

                wait_seconds = min(2 ** attempt, 30)

                print(
                    f"Temporary API error "
                    f"{response.status_code}. "
                    f"Retrying in {wait_seconds}s..."
                )

                time.sleep(wait_seconds)
                continue

            response.raise_for_status()

        except requests.RequestException as exc:

            if attempt == max_retries:
                raise RuntimeError(
                    f"Request failed after "
                    f"{max_retries} attempts."
                ) from exc

            wait_seconds = min(2 ** attempt, 30)

            print(
                f"Request error: {exc}. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

    raise RuntimeError("API request failed unexpectedly.")

In [0]:
# Test one indicator first

TEST_INDICATOR = "NY.GDP.MKTP.KD.ZG"

test_url = (
    f"{WORLD_BANK_API_BASE_URL}/indicator/"
    f"{TEST_INDICATOR}"
)

test_response = get_json_with_retry(
    test_url,
    params={"format": "json"}
)

print(f"Response type: {type(test_response)}")

if isinstance(test_response, list):
    print(f"Top-level elements: {len(test_response)}")

Response type: <class 'list'>
Top-level elements: 2


In [0]:
# inspect the actual record:

if (
    not isinstance(test_response, list)
    or len(test_response) < 2
    or not test_response[1]
):
    raise RuntimeError(
        f"No metadata returned for {TEST_INDICATOR}"
    )

sample_indicator = test_response[1][0]

for key, value in sample_indicator.items():
    print(f"{key}: {value}")

id: NY.GDP.MKTP.KD.ZG
name: GDP growth (annual %)
unit: 
source: {'id': '2', 'value': 'World Development Indicators'}
sourceNote: Gross domestic product is the total income earned through the production of goods and services in an economic territory during an accounting period. It can be measured in three different ways: using either the expenditure approach, the income approach, or the production approach. This indicator denotes the percentage change over each previous year of the constant price (base year 2015) series in United States dollars.
sourceOrganization: Country official statistics, National Statistical Organizations and/or Central Banks;
National Accounts data files, Organisation for Economic Co-operation and Development (OECD);
Staff estimates, World Bank (WB)
topics: [{'id': '3', 'value': 'Economy & Growth'}]


In [0]:
# Function to fetch one indicator


def fetch_indicator_metadata(indicator_code):

    url = (
        f"{WORLD_BANK_API_BASE_URL}/indicator/"
        f"{indicator_code}"
    )

    response = get_json_with_retry(
        url,
        params={"format": "json"}
    )

    if (
        not isinstance(response, list)
        or len(response) < 2
        or not response[1]
    ):
        return None

    return response[1][0]

In [0]:

indicator_results = []

for indicator_code, configured_name in INDICATORS.items():

    print(f"Checking {indicator_code}...")

    try:
        metadata = fetch_indicator_metadata(
            indicator_code
        )

        if metadata is None:

            indicator_results.append({
                "indicator_code": indicator_code,
                "configured_name": configured_name,
                "is_valid": False,
                "metadata": None,
                "error_message": "No metadata returned"
            })

            print("  INVALID - no metadata returned")

        else:

            indicator_results.append({
                "indicator_code": indicator_code,
                "configured_name": configured_name,
                "is_valid": True,
                "metadata": metadata,
                "error_message": None
            })

            print(
                f"  VALID - "
                f"{metadata.get('name')}"
            )

    except Exception as exc:

        indicator_results.append({
            "indicator_code": indicator_code,
            "configured_name": configured_name,
            "is_valid": False,
            "metadata": None,
            "error_message": str(exc)
        })

        print(f"  ERROR - {exc}")

Checking NY.GDP.MKTP.KD.ZG...
  VALID - GDP growth (annual %)
Checking NY.GDP.PCAP.KD.ZG...
  VALID - GDP per capita growth (annual %)
Checking NY.GDP.MKTP.CD...
  VALID - GDP (current US$)
Checking NY.GDP.PCAP.CD...
  VALID - GDP per capita (current US$)
Checking FP.CPI.TOTL.ZG...
  VALID - Inflation, consumer prices (annual %)
Checking NE.TRD.GNFS.ZS...
  VALID - Trade (% of GDP)
Checking NE.EXP.GNFS.ZS...
  VALID - Exports of goods and services (% of GDP)
Checking NE.IMP.GNFS.ZS...
  VALID - Imports of goods and services (% of GDP)
Checking NE.GDI.TOTL.ZS...
  VALID - Gross capital formation (% of GDP)
Checking GC.XPN.TOTL.GD.ZS...
  VALID - Expense (% of GDP)
Checking GC.REV.XGRT.GD.ZS...
  VALID - Revenue, excluding grants (% of GDP)
Checking GC.DOD.TOTL.GD.ZS...
  VALID - Central government debt, total (% of GDP)
Checking NY.GDS.TOTL.ZS...
  VALID - Gross domestic savings (% of GDP)
Checking BX.KLT.DINV.WD.GD.ZS...
  VALID - Foreign direct investment, net inflows (% of GDP)
Check

In [0]:
# Validation summary


valid_indicators = [
    result
    for result in indicator_results
    if result["is_valid"]
]

invalid_indicators = [
    result
    for result in indicator_results
    if not result["is_valid"]
]

print("=" * 65)
print("INDICATOR VALIDATION")
print("=" * 65)

print(f"Configured: {len(INDICATORS)}")
print(f"Valid:      {len(valid_indicators)}")
print(f"Invalid:    {len(invalid_indicators)}")

if invalid_indicators:

    print("\nInvalid indicators:")

    for result in invalid_indicators:
        print(
            f"{result['indicator_code']} -> "
            f"{result['error_message']}"
        )

INDICATOR VALIDATION
Configured: 15
Valid:      15
Invalid:    0


In [0]:
# Look at official names


for result in valid_indicators:

    metadata = result["metadata"]

    print(
        f"{result['indicator_code']}\n"
        f"Configured: {result['configured_name']}\n"
        f"World Bank: {metadata.get('name')}\n"
    )

NY.GDP.MKTP.KD.ZG
Configured: GDP growth (annual %)
World Bank: GDP growth (annual %)

NY.GDP.PCAP.KD.ZG
Configured: GDP per capita growth (annual %)
World Bank: GDP per capita growth (annual %)

NY.GDP.MKTP.CD
Configured: GDP (current US$)
World Bank: GDP (current US$)

NY.GDP.PCAP.CD
Configured: GDP per capita (current US$)
World Bank: GDP per capita (current US$)

FP.CPI.TOTL.ZG
Configured: Inflation, consumer prices (annual %)
World Bank: Inflation, consumer prices (annual %)

NE.TRD.GNFS.ZS
Configured: Trade (% of GDP)
World Bank: Trade (% of GDP)

NE.EXP.GNFS.ZS
Configured: Exports of goods and services (% of GDP)
World Bank: Exports of goods and services (% of GDP)

NE.IMP.GNFS.ZS
Configured: Imports of goods and services (% of GDP)
World Bank: Imports of goods and services (% of GDP)

NE.GDI.TOTL.ZS
Configured: Gross capital formation (% of GDP)
World Bank: Gross capital formation (% of GDP)

GC.XPN.TOTL.GD.ZS
Configured: Expense (% of GDP)
World Bank: Expense (% of GDP)

GC.RE

In [0]:
# Create Bronze records


ingestion_timestamp = datetime.now(timezone.utc)

bronze_records = []

for result in indicator_results:

    metadata = result["metadata"] or {}

    source = metadata.get("source") or {}

    topics = metadata.get("topics") or []

    topic_values = []

    for topic in topics:
        if isinstance(topic, dict):
            topic_value = topic.get("value")

            if topic_value:
                topic_values.append(topic_value)

    bronze_records.append({
        "indicator_code": result["indicator_code"],
        "configured_name": result["configured_name"],

        "indicator_name": metadata.get("name"),

        "unit": metadata.get("unit"),

        "source_id": source.get("id"),
        "source_name": source.get("value"),

        "source_note": metadata.get("sourceNote"),

        "source_organization":
            metadata.get("sourceOrganization"),

        "topics_json": json.dumps(
            topics,
            ensure_ascii=False
        ),

        "topic_names": topic_values,

        "is_valid": result["is_valid"],

        "validation_error":
            result["error_message"],

        "source_system":
            "World Bank Indicators API",

        "source_endpoint":
            (
                f"{WORLD_BANK_API_BASE_URL}/indicator/"
                f"{result['indicator_code']}"
            ),

        "ingested_at":
            ingestion_timestamp
    })

In [0]:
# Define the schema


metadata_schema = T.StructType([

    T.StructField(
        "indicator_code",
        T.StringType(),
        False
    ),

    T.StructField(
        "configured_name",
        T.StringType(),
        True
    ),

    T.StructField(
        "indicator_name",
        T.StringType(),
        True
    ),

    T.StructField(
        "unit",
        T.StringType(),
        True
    ),

    T.StructField(
        "source_id",
        T.StringType(),
        True
    ),

    T.StructField(
        "source_name",
        T.StringType(),
        True
    ),

    T.StructField(
        "source_note",
        T.StringType(),
        True
    ),

    T.StructField(
        "source_organization",
        T.StringType(),
        True
    ),

    T.StructField(
        "topics_json",
        T.StringType(),
        True
    ),

    T.StructField(
        "topic_names",
        T.ArrayType(T.StringType()),
        True
    ),

    T.StructField(
        "is_valid",
        T.BooleanType(),
        False
    ),

    T.StructField(
        "validation_error",
        T.StringType(),
        True
    ),

    T.StructField(
        "source_system",
        T.StringType(),
        False
    ),

    T.StructField(
        "source_endpoint",
        T.StringType(),
        False
    ),

    T.StructField(
        "ingested_at",
        T.TimestampType(),
        False
    )
])

In [0]:
# Create the DataFrame


indicator_metadata_df = spark.createDataFrame(
    bronze_records,
    schema=metadata_schema
)

display(
    indicator_metadata_df.select(
        "indicator_code",
        "indicator_name",
        "unit",
        "source_name",
        "topic_names",
        "is_valid",
        "validation_error"
    )
)

indicator_code,indicator_name,unit,source_name,topic_names,is_valid,validation_error
NY.GDP.MKTP.KD.ZG,GDP growth (annual %),,World Development Indicators,List(Economy & Growth),true,null
NY.GDP.PCAP.KD.ZG,GDP per capita growth (annual %),,World Development Indicators,List(Economy & Growth),true,null
NY.GDP.MKTP.CD,GDP (current US$),,World Development Indicators,List(Economy & Growth),true,null
NY.GDP.PCAP.CD,GDP per capita (current US$),,World Development Indicators,List(Economy & Growth),true,null
FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",,World Development Indicators,"List(Economy & Growth, Financial Sector )",true,null
NE.TRD.GNFS.ZS,Trade (% of GDP),,World Development Indicators,"List(Economy & Growth, Trade)",true,null
NE.EXP.GNFS.ZS,Exports of goods and services (% of GDP),,World Development Indicators,"List(Economy & Growth, Trade)",true,null
NE.IMP.GNFS.ZS,Imports of goods and services (% of GDP),,World Development Indicators,"List(Economy & Growth, Trade)",true,null
NE.GDI.TOTL.ZS,Gross capital formation (% of GDP),,World Development Indicators,List(Economy & Growth),true,null
GC.XPN.TOTL.GD.ZS,Expense (% of GDP),,World Development Indicators,"List(Economy & Growth, Public Sector )",true,null


In [0]:
# Validate uniqueness


total_rows = indicator_metadata_df.count()

duplicate_codes = (
    indicator_metadata_df
    .groupBy("indicator_code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Configured indicators: {len(INDICATORS)}")
print(f"Metadata rows: {total_rows}")
print(f"Duplicate codes: {duplicate_codes}")

if total_rows != len(INDICATORS):
    raise RuntimeError(
        "Metadata row count does not match "
        "configured indicator count."
    )

if duplicate_codes > 0:
    raise RuntimeError(
        "Duplicate indicator codes detected."
    )

print("Indicator metadata validation passed.")

Configured indicators: 15
Metadata rows: 15
Duplicate codes: 0
Indicator metadata validation passed.


In [0]:
# Write Bronze Delta


(
    indicator_metadata_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(INDICATOR_METADATA_TABLE)
)

print(
    f"Saved: {INDICATOR_METADATA_TABLE}"
)

Saved: worldbank_ai.bronze.indicator_metadata_raw


In [0]:
# Read it back


saved_metadata_df = spark.table(
    INDICATOR_METADATA_TABLE
)

saved_count = saved_metadata_df.count()

print(f"Expected rows: {len(INDICATORS)}")
print(f"Saved rows:    {saved_count}")

if saved_count != len(INDICATORS):
    raise RuntimeError(
        "Indicator metadata write validation failed."
    )

print("Bronze metadata write validated.")

Expected rows: 15
Saved rows:    15
Bronze metadata write validated.


In [0]:
# Show active indicators


active_indicators_df = (
    saved_metadata_df
    .filter(F.col("is_valid") == True)
    .select(
        "indicator_code",
        "indicator_name",
        "unit",
        "source_name",
        "topic_names"
    )
    .orderBy("indicator_code")
)

display(active_indicators_df)

print(
    f"Indicators approved for observation ingestion: "
    f"{active_indicators_df.count()}"
)

indicator_code,indicator_name,unit,source_name,topic_names
BN.CAB.XOKA.GD.ZS,Current account balance (% of GDP),,World Development Indicators,List(Economy & Growth)
BX.KLT.DINV.WD.GD.ZS,"Foreign direct investment, net inflows (% of GDP)",,World Development Indicators,"List(Economy & Growth, Financial Sector , Climate Change)"
FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",,World Development Indicators,"List(Economy & Growth, Financial Sector )"
GC.DOD.TOTL.GD.ZS,"Central government debt, total (% of GDP)",,World Development Indicators,"List(Economy & Growth, Public Sector )"
GC.REV.XGRT.GD.ZS,"Revenue, excluding grants (% of GDP)",,World Development Indicators,"List(Economy & Growth, Public Sector )"
GC.XPN.TOTL.GD.ZS,Expense (% of GDP),,World Development Indicators,"List(Economy & Growth, Public Sector )"
NE.EXP.GNFS.ZS,Exports of goods and services (% of GDP),,World Development Indicators,"List(Economy & Growth, Trade)"
NE.GDI.TOTL.ZS,Gross capital formation (% of GDP),,World Development Indicators,List(Economy & Growth)
NE.IMP.GNFS.ZS,Imports of goods and services (% of GDP),,World Development Indicators,"List(Economy & Growth, Trade)"
NE.TRD.GNFS.ZS,Trade (% of GDP),,World Development Indicators,"List(Economy & Growth, Trade)"


Indicators approved for observation ingestion: 15
